# View-embedding trainer (Colab) — feeds on the view_dataset export

Upload THREE files to the Colab session (e.g. drag into /content/):
  1. view_dataset.parquet   (the per-frame view manifest)
  2. view_dataset.json      (sidecar: content_hash, params, stats)
  3. oceanside_clip.mp4      (the SAME clip the labeler used)

This script is SELF-CONTAINED — only torch/torchvision/opencv/pandas/numpy
(all preinstalled in Colab). It reimplements the essential ViewFrameReader
decode logic so you do NOT need to pip-install the soccer_vision package.

What the export gives you (per frame row): view_id (smoothed pseudo-label),
view_key ("game:view_id" — the contrastive CLASS), confidence/weight (QC/loss
weight), margin + ambiguous (pan-transition flag), view_second (runner-up view
for soft labels), split (train/val), t_seconds. See stats in the .json.

In [ ]:
import json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import torch
from torch.utils.data import Dataset, DataLoader

EXPORT_DIR = Path("/content")                     # where you dropped the 2 export files
VIDEO      = Path("/content/oceanside_clip.mp4")  # the uploaded clip
IMG_SIZE   = 224                                  # backbone input
DOWNSCALE  = 0.5                                  # decode at half-res first (cheap), then resize

manifest = pd.read_parquet(EXPORT_DIR / "view_dataset.parquet")
meta = json.loads((EXPORT_DIR / "view_dataset.json").read_text())
print("rows:", len(manifest), "| views:", meta["digest"]["n_views"],
      "| train/val:", meta["stats"]["n_train"], "/", meta["stats"]["n_val"])
print("stats:", {k: meta["stats"][k] for k in ("switch_rate", "n_ambiguous",
                                                "confidence_median")})

In [ ]:
def _content_hash(video_path, edge_bytes=1 << 20):
    """Path/mtime-independent fingerprint — must match meta['video']['content_hash']."""
    p = Path(video_path); size = p.stat().st_size
    h = hashlib.sha1(); h.update(str(size).encode())
    with open(p, "rb") as f:
        h.update(f.read(edge_bytes))
        if size > edge_bytes:
            f.seek(max(0, size - edge_bytes)); h.update(f.read(edge_bytes))
    return h.hexdigest()[:16]

# Sanity: confirm the uploaded clip is the one the manifest was built for.
got, want = _content_hash(VIDEO), meta["video"]["content_hash"]
assert got == want, f"wrong/re-encoded video: {got} != {want}"
print("content hash OK:", got)

In [ ]:
class ViewFrameDataset(Dataset):
    """Decodes frames on demand from the mp4 using the manifest.

    Contrastive labels: `view_id` (int) / `view_key` (str). Rows with view_id == -1
    (unlabelable low-keypoint frames) are dropped. Each worker opens its OWN cv2
    capture (a VideoCapture is not picklable) via lazy init keyed on worker id.
    Returns (img[C,H,W] float in RGB, view_id:int, weight:float).
    """
    def __init__(self, manifest, video_path, split=None, img_size=224, downscale=0.5,
                 drop_unassigned=True):
        m = manifest
        if split is not None:
            m = m[m["split"] == split]
        if drop_unassigned:
            m = m[m["view_id"] != -1]
        self.rows = m.reset_index(drop=True)
        self.video_path = str(video_path)
        self.img_size = img_size
        self.downscale = downscale
        self._cap = None
        self._pos = 0
        # contiguous 0..K-1 class ids for a classification/contrastive head
        self.view_ids = sorted(self.rows["view_id"].unique().tolist())
        self.view_to_class = {v: i for i, v in enumerate(self.view_ids)}

    def __len__(self):
        return len(self.rows)

    def _cap_open(self):
        # lazy per-worker capture (survives DataLoader fork with num_workers>0)
        if self._cap is None:
            self._cap = cv2.VideoCapture(self.video_path)
            self._pos = 0
        return self._cap

    def _decode(self, frame_idx):
        cap = self._cap_open()
        if frame_idx < self._pos:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx); self._pos = frame_idx
        while self._pos < frame_idx:
            if not cap.grab(): break
            self._pos += 1
        ok, frame = cap.read(); self._pos += 1
        if not ok:
            raise IndexError(f"decode failed at frame {frame_idx}")
        return frame  # BGR

    def __getitem__(self, i):
        row = self.rows.iloc[i]
        frame = self._decode(int(row["frame"]))
        if self.downscale != 1.0:
            frame = cv2.resize(frame, None, fx=self.downscale, fy=self.downscale)
        frame = cv2.resize(frame, (self.img_size, self.img_size))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)                 # torch wants RGB
        img = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0
        return img, self.view_to_class[int(row["view_id"])], float(row["weight"])


def _worker_init(_):
    # ensure each worker gets a fresh capture (defensive; lazy init also handles it)
    info = torch.utils.data.get_worker_info()
    if info is not None:
        info.dataset._cap = None


train_ds = ViewFrameDataset(manifest, VIDEO, split="train", img_size=IMG_SIZE, downscale=DOWNSCALE)
val_ds   = ViewFrameDataset(manifest, VIDEO, split="val",   img_size=IMG_SIZE, downscale=DOWNSCALE)
print("train:", len(train_ds), "val:", len(val_ds), "classes:", len(train_ds.view_ids))

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2,
                      worker_init_fn=_worker_init, drop_last=True)
val_dl   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2,
                      worker_init_fn=_worker_init)

# smoke test one batch
xb, yb, wb = next(iter(train_dl))
print("batch:", xb.shape, xb.dtype, "labels:", yb[:8].tolist(), "weights:", wb[:4].tolist())

## Starter model — backbone → embedding, trained on the view pseudo-labels

Simplest pretext that yields a clustering-ready embedding: classify the view_id
(cross-entropy) on a pretrained backbone; take the penultimate features as the
low-dim vector. Swap in a SupCon / triplet loss later for a metric embedding;
swap in a masked-autoencoder once tracking boxes exist (then `n_boxes>0` rows
carry a player keep_mask you can union into the MAE mask).

In [ ]:
import torch.nn as nn
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"
EMB_DIM = 128
n_classes = len(train_ds.view_ids)

backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
backbone.fc = nn.Identity()                       # -> 512-d features
model = nn.Sequential(
    backbone,
    nn.Linear(512, EMB_DIM), nn.ReLU(),           # the embedding you cluster later
).to(device)
head = nn.Linear(EMB_DIM, n_classes).to(device)   # view-id classification pretext

opt = torch.optim.Adam(list(model.parameters()) + list(head.parameters()), lr=1e-4)
ce = nn.CrossEntropyLoss(reduction="none")

def run_epoch(dl, train=True):
    model.train(train); head.train(train)
    tot, correct, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for xb, yb, wb in dl:
            xb, yb, wb = xb.to(device), yb.to(device), wb.to(device)
            emb = model(xb); logits = head(emb)
            loss = (ce(logits, yb) * wb).mean()    # weight by assignment confidence
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += float(loss) * len(yb)
            correct += int((logits.argmax(1) == yb).sum()); tot += len(yb)
    return loss_sum / max(tot, 1), correct / max(tot, 1)

for epoch in range(5):
    tl, ta = run_epoch(train_dl, True)
    vl, va = run_epoch(val_dl, False)
    print(f"epoch {epoch}: train loss {tl:.3f} acc {ta:.2f} | val loss {vl:.3f} acc {va:.2f}")

`model(x)` now yields a 128-d embedding per frame. Next steps you own:
 - dump embeddings for all frames, UMAP + cluster, compare to view_id (should recover them).
 - upgrade the loss to SupCon/triplet for a true metric space, or MAE once masks exist.
 - to scale cross-game, re-export each game and concatenate manifests (view_key is
   game-scoped, so views never collide across clips).